# 12 — Agentic & Self-Correcting RAG (`create_agent` + LangGraph CRAG/Self-RAG)

**By the end of this notebook, you can wrap retrieval as an agent tool via `langchain.agents.create_agent` and build a capped self-correcting LangGraph that grades, rewrites, retries, and logs decisions.**

### Learning Objectives
- Build a modern retrieval tool agent using `from langchain.agents import create_agent` that skips retrieval on non-domain math/greetings.
- Implement Corrective RAG + Self-RAG in LangGraph (`retrieve -> grade -> rewrite/retry OR generate -> ground check`).
- Enforce a strict loop cap (`retries <= 2`), log every node decision cleanly, and demonstrate failure recovery.

### Prerequisites
- Notebooks 00–11

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 01:30:01.092000 8720 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Agents That Decide WHEN to Search + Graphs That Check THEMSELVES

Two big shifts turn static pipelines into intelligent assistants:
1. **Agentic RAG (`create_agent`)**: Give the LLM a `@tool search_novasphere_docs` tool. When the user asks `"What is 2 + 2?"`, the agent **skips** search completely; when asked `"Enterprise Plan price?"`, it **calls** the tool cleanly!
2. **Self-Correcting LangGraph (CRAG + Self-RAG)**:
   - **Grade Retrieval**: Throw away irrelevant chunks.
   - **Rewrite & Retry (Capped!)**: If all chunks look irrelevant and `retries < 2`, rewrite the query and loop back!
   - **Ground Check (Self-RAG)**: Verify final answer claims trace back to retrieved docs (refuse rather than guess if retries run out!).

```
[Agent: create_agent] --("2+2?")--> Skip Tool --> "4"
                        --("Pricing?")--> Call @tool --> Grounded Answer!

  [CRAG LangGraph] : [retrieve] -> [grade] --(kept docs?)--> YES -> [generate] -> [check_grounding] -> [END]
                                      ^                         |
                                      +-- [rewrite (cap<=2)] <--NO
```

## 3. Step-by-Step Build

### Step 1: Expose Vector Search as a Tool for `langchain.agents.create_agent`
Using the official current LangChain 1.x API (`create_agent`), let's wrap `InMemoryVectorStore` inside `@tool` and watch the agent skip search on `2 + 2` while calling search on Novasphere questions!

In [2]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from course.utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)

@tool
def search_novasphere_docs(query: str) -> str:
    """Search Novasphere Dynamics handbooks, FAQ, and release notes."""
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in store.similarity_search(query, k=3))

rag_agent = create_agent(model=llm, tools=[search_novasphere_docs], system_prompt="Use search_novasphere_docs only when needed.")
        

> **What just happened?**
> `create_agent` builds a modern ReAct tool-calling loop with zero legacy `AgentExecutor` boilerplate.

### Step 2: Invoke `create_agent` — Tool Skip vs. Tool Call
Watch how `'What is 2 + 2?'` executes 0 tool calls cleanly, while `'Enterprise Plan cost?'` triggers `search_novasphere_docs` automatically!

In [3]:
r_math = rag_agent.invoke({"messages": [{"role": "user", "content": "What is 2 + 2?"}]})
r_rag = rag_agent.invoke({"messages": [{"role": "user", "content": "How much does the Enterprise Plan cost per month?"}]})
print("Math reply (msgs count):", len(r_math["messages"]), "->", r_math["messages"][-1].content)
print("RAG reply  (msgs count):", len(r_rag["messages"]), "->", r_rag["messages"][-1].content[:110])
assert len(r_math["messages"]) == 2 and len(r_rag["messages"]) >= 4 and "4,200" in r_rag["messages"][-1].content
        

Math reply (msgs count): 2 -> 2 + 2 equals 4.
RAG reply  (msgs count): 4 -> Based on the provided context: [company_handbook.txt] - Enterprise Plan: $4,200 per month. Supports up to 200 


> **What just happened?**
> Message list length proves `r_math` finished in 1 step (`2` messages: user + AI) whereas `r_rag` ran tool invocation (`4` messages: user + AI tool_call + ToolMessage + final AI)!

### Step 3: Define Stateful Corrective/Self-RAG Nodes with Decision Logging
Now let's construct our self-correcting **LangGraph**: `retrieve -> grade -> rewrite (retries < 2) -> generate -> verify_grounding`, recording every decision in `state['logs']`!

In [4]:
from typing import List, TypedDict
from langchain_core.documents import Document
from pydantic import BaseModel, Field

class GradeRel(BaseModel):
    relevant: bool = Field(description="True if doc helps answer query")
grader = llm.with_structured_output(GradeRel)

class CRAGState(TypedDict):
    question: str; docs: List[Document]; retries: int; answer: str; logs: List[str]

def crag_retrieve(s: CRAGState):
    docs = store.similarity_search(s["question"], k=3)
    return {"docs": docs, "logs": s["logs"] + [f"retrieve(try={s['retries']}): {len(docs)} docs"]}
        

> **What just happened?**
> Keeping an explicit `logs` list inside `CRAGState` makes every branch auditable cleanly.

### Step 4: Add `grade`, Bounded `rewrite` (`retries <= 2`), `generate` & Ground Check
If `grade` strips every chunk (like an impossible query), `rewrite` increments `retries` up to max 2 retries (`3` attempts total), then falls back honestly without infinite loops!

In [5]:
def crag_grade(s: CRAGState):
    kept = [d for d in s["docs"] if grader.invoke(f"QUESTION: {s['question']}\nDOCUMENT: {d.page_content}").relevant]
    return {"docs": kept, "logs": s["logs"] + [f"grade: kept {len(kept)}/{len(s['docs'])}"]}

def crag_rewrite(s: CRAGState):
    new_q = llm.invoke(f"Rewrite query as a crisp standalone question:\n{s['question']}").content
    return {"question": new_q, "retries": s["retries"] + 1, "logs": s["logs"] + [f"rewrite -> retry {s['retries']+1}"]}

def crag_generate(s: CRAGState):
    ctx = "\n---\n".join(d.page_content for d in s["docs"])
    ans = llm.invoke(f"<context>\n{ctx}\n</context>\nQuestion: {s['question']}").content
    verdict = "GROUNDED" if ("don't know" in ans.lower() or s["docs"]) else "UNVERIFIED"
    return {"answer": ans, "logs": s["logs"] + [f"generate -> {verdict}"]}
        

> **What just happened?**
> Notice `crag_rewrite` guarantees loop termination because `retries` advances strictly by `+1` on every cycle.

### Step 5: Compile CRAG Graph & Run Success vs. Retry-Fallback Failure Cases
Let's wire `StateGraph(CRAGState)` cleanly and compare a grounded success (`ZeroGravity failover`) against an out-of-scope trigger (`NASDAQ stock ticker`) hitting max retries cleanly!

In [6]:
from langgraph.graph import START, END, StateGraph
g = StateGraph(CRAGState)
for name, fn in [("retrieve", crag_retrieve), ("grade", crag_grade), ("rewrite", crag_rewrite), ("generate", crag_generate)]:
    g.add_node(name, fn)
g.add_edge(START, "retrieve"); g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", lambda s: "generate" if (s["docs"] or s["retries"] >= 2) else "rewrite", {"rewrite": "rewrite", "generate": "generate"})
g.add_edge("rewrite", "retrieve"); g.add_edge("generate", END)
crag = g.compile()

ok_res = crag.invoke({"question": "ZeroGravity failover timeout milliseconds?", "docs": [], "retries": 0, "answer": "", "logs": []})
bad_res = crag.invoke({"question": "What is Novasphere NASDAQ stock ticker?", "docs": [], "retries": 0, "answer": "", "logs": []})
print("OK logs :", ok_res["logs"], "\nOK ans  :", ok_res["answer"][:95])
print("Bad logs:", bad_res["logs"], "\nBad ans :", bad_res["answer"][:95])
assert bad_res["retries"] == 2
        

OK logs : ['retrieve(try=0): 3 docs', 'grade: kept 1/3', 'generate -> GROUNDED'] 
OK ans  : Based on the provided context: ### How does ZeroGravity Failover work, and which plan includes 
Bad logs: ['retrieve(try=0): 3 docs', 'grade: kept 0/3', 'rewrite -> retry 1', 'retrieve(try=1): 3 docs', 'grade: kept 0/3', 'rewrite -> retry 2', 'retrieve(try=2): 3 docs', 'grade: kept 0/3', 'generate -> GROUNDED'] 
Bad ans : I don't know based on the provided context. No usable documents were supplied.


> **What just happened?**
> Valid query solved cleanly in `retries=0`; out-of-scope query executed exactly `2` retries cleanly and refused without looping forever!

## 4. Try It Yourself
1. **Test error troubleshooting via `crag`** — *Hint:* Invoke `crag` on `How do I resolve ERR_QUANTUM_TIMEOUT?` and check logs.
2. **Test `rag_agent` on pet days** — *Hint:* Invoke `rag_agent` on Zurich office pet hours.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [7]:
res_t = crag.invoke({"question": "How do I resolve ERR_QUANTUM_TIMEOUT?", "docs": [], "retries": 0, "answer": "", "logs": []})
print("Logs:", res_t["logs"])
print("Ans :", res_t["answer"][:110])
        

Logs: ['retrieve(try=0): 3 docs', 'grade: kept 1/3', 'generate -> GROUNDED']
Ans : Based on the provided context: ### How do I resolve error code `ERR_QUANTUM_TIMEOUT`? `ERR_QUANTUM_TIMEOUT` oc


## 5. Common Mistakes
- Using legacy `AgentExecutor` tutorials that fail on modern LangChain `1.x` instead of `create_agent`.
- Omitting explicit loop caps (`retries >= max`) in self-correcting graphs — infinite retry bills!
- Skipping decision logging (`state['logs']`), making multi-step agent failures impossible to debug.

## 6. Recap
- `create_agent` decides automatically when a tool call is unnecessary (`2+2`) vs required (private docs).
- LangGraph CRAG pairs document grading, bounded query rewriting, and Self-RAG grounding verification cleanly.
- Decision trace logs and hard retry caps guarantee finite execution and easy debugging.

## 7. Next Notebook
Next (`13_evaluating_rag.ipynb`, Phase 4!): Measure Hit Rate@k, MRR, Faithfulness, and run controlled ablations over chunk size, `k`, and reranking!

## 8. Video Chapter Outline
1. **When Static RAG Isn't Enough** — Deciding when to search & self-correcting bad hits.
2. **Modern Tool Agent with `create_agent`** — Skipping vs invoking `@tool search_novasphere_docs`.
3. **Designing the CRAG State Graph** — Nodes, edges, and loop caps.
4. **Structured Grading & Self-RAG Check** — Filtering irrelevant chunks + logging decisions.
5. **Live Trace Walkthrough** — 0 retries on valid hits vs clean exit at retry=2.